In [ ]:
class TicTacToe:
    def __init__(self, M: int, N: int, K: int):
        """
        Setup M x N board. Need K marks to win.
        """
        self.rows = M
        self.cols = N
        self.k = K
        self.board = [[0] * N for _ in range(M)]
        self.current_player = 1  # Player 1 goes first
        self.game_over = False

    def move(self, i: int, j: int) -> int:
        """
        Place mark at (i, j). Print board and check winner.
        Returns: 0 (no winner), 1 (player 1 wins), 2 (player 2 wins)
        """
        if self.game_over:
            print("Game is already over!")
            return 0

        if not (0 <= i < self.rows and 0 <= j < self.cols):
            print(f"Invalid move: ({i}, {j}) is out of bounds. Try again.")
            return -1

        # --- NEW: Availability check ---
        if self.board[i][j] != 0:
            print(f"Invalid move: Cell ({i}, {j}) is already occupied. Try again.")
            return -1

        # Place the mark
        self.board[i][j] = self.current_player

        # Print the board
        self._print_board()

        # Check for winner
        if self._check_win(i, j):
            print(f"Player {self.current_player} won")
            self.game_over = True
            return self.current_player

              # Check for draw
        if self._is_board_full():
            print("Draw")
            self.game_over = True
            return 0

        # Switch player
        self.current_player = 3 - self.current_player  # Toggle between 1 and 2

        return 0

    def _print_board(self):
        """Print the board visually."""
        symbols = {0: '   ', 1: ' X ', 2: ' O '}
        print("\nBoard:")
        for row in self.board:
            print('|' + '|'.join(symbols[cell] for cell in row) + '|')
        print()

    def _check_win(self, row: int, col: int) -> bool:
        """
        Check if the last move at (row, col) caused a win.
        """
        player = self.current_player

        # Check all 4 directions
        directions = [
            (0, 1),   # Horizontal
            (1, 0),   # Vertical
            (1, 1),   # Main diagonal
            (1, -1)   # Anti-diagonal
        ]

        for dr, dc in directions:
            count = 1  # Count the current spot

            # Count going forward
            count += self._count_consecutive(row, col, dr, dc, player)

            # Count going backward
            count += self._count_consecutive(row, col, -dr, -dc, player)

            if count >= self.k:
                return True

        return False

    def _count_consecutive(self, row: int, col: int, dr: int, dc: int, player: int) -> int:
        """
        Count how many marks match in one direction.
        """
        count = 0
        r, c = row + dr, col + dc

        while 0 <= r < self.rows and 0 <= c < self.cols and self.board[r][c] == player:
            count += 1
            r += dr
            c += dc

        return count

# Example usage
if __name__ == "__main__":
    game = TicTacToe(3, 4, 3)

    game.move(0, 0)  # Player 1
    game.move(0, 1)  # Player 2
    game.move(1, 0)  # Player 1
    game.move(1, 1)  # Player 2
    game.move(2, 0)  # Player 1 wins



Board:
| X |   |   |   |
|   |   |   |   |
|   |   |   |   |


Board:
| X | O |   |   |
|   |   |   |   |
|   |   |   |   |


Board:
| X | O |   |   |
| X |   |   |   |
|   |   |   |   |


Board:
| X | O |   |   |
| X | O |   |   |
|   |   |   |   |


Board:
| X | O |   |   |
| X | O |   |   |
| X |   |   |   |

Player 1 won


In [ ]:
## Follow up with AI :
import random
from typing import Optional, Tuple

class TicTacToeWithAI:
    def __init__(self, M: int, N: int, K: int, isAI: bool = False):
        """
        Setup board. If isAI is True, Player 2 is computer.
        """
        self.rows = M
        self.cols = N
        self.k = K
        self.board = [[0] * N for _ in range(M)]
        self.current_player = 1  # Player 1 starts (human)
        self.game_over = False
        self.is_ai_enabled = isAI
        self.ai_player = 2  # AI is Player 2

    def move(self, i: int, j: int) -> int:
        """
        Human move at (i, j). If AI is on, AI moves immediately after.
        """
        if self.game_over:
            print("Game is already over!")
            return 0

        # Place the mark
        result = self._make_move(i, j)

        if result != 0:  # Game ended (win or draw)
            return result

        # If AI is enabled and it's now AI's turn, make AI move
        if self.is_ai_enabled and self.current_player == self.ai_player:
            return self._ai_move()

        return 0

    def _make_move(self, i: int, j: int) -> int:
        """
        Helper to place mark and check result.
        """
        # Place the mark
        self.board[i][j] = self.current_player

        # Print the board
        self._print_board()

        # Check for winner
        if self._check_win(i, j):
            print(f"Player {self.current_player} won")
            self.game_over = True
            return self.current_player

        # Check for draw
        if self._is_board_full():
            print("Draw")
            self.game_over = True
            return -1

        # Switch player
        self.current_player = 3 - self.current_player

        return 0

    def _ai_move(self) -> int:
        """
        AI logic: Win -> Block -> Random.
        """
        print(f"AI (Player {self.ai_player}) is thinking...")

        # Strategy 1: Try to win
        move = self._find_winning_move(self.ai_player)
        if move:
            print(f"AI plays at ({move[0]}, {move[1]})")
            return self._make_move(move[0], move[1])

        # Strategy 2: Block opponent from winning
        opponent = 3 - self.ai_player
        move = self._find_winning_move(opponent)
        if move:
            print(f"AI blocks at ({move[0]}, {move[1]})")
            return self._make_move(move[0], move[1])

        # Strategy 3: Random move
        move = self._get_random_empty_cell()
        if move:
            print(f"AI plays at ({move[0]}, {move[1]})")
            return self._make_move(move[0], move[1])

        return 0

    def _find_winning_move(self, player: int) -> Optional[Tuple[int, int]]:
        """
        Look for a move that wins immediately for 'player'.
        """
        for i in range(self.rows):
            for j in range(self.cols):
                if self.board[i][j] == 0:  # Empty cell
                    # Try this move
                    self.board[i][j] = player
                    wins = self._check_win(i, j, player)
                    self.board[i][j] = 0  # Undo
                    if wins:
                        return (i, j)

        return None

    def _get_random_empty_cell(self) -> Optional[Tuple[int, int]]:
        """
        Pick any empty spot.
        """
        empty_cells = []
        for i in range(self.rows):
            for j in range(self.cols):
                if self.board[i][j] == 0:
                    empty_cells.append((i, j))

        return random.choice(empty_cells) if empty_cells else None

    def _is_board_full(self) -> bool:
        """Check if board is full."""
        for row in self.board:
            if 0 in row:
                return False
        return True

    def _print_board(self):
        """Print the board."""
        symbols = {0: '   ', 1: ' X ', 2: ' O '}
        print("\nBoard:")
        for row in self.board:
            print('|' + '|'.join(symbols[cell] for cell in row) + '|')
        print()

    def _check_win(self, row: int, col: int, player: Optional[int] = None) -> bool:
        """
        Check if 'player' won after a move at (row, col).
        """
        if player is None:
            player = self.current_player

        # Check all 4 directions
        directions = [
            (0, 1),   # Horizontal
            (1, 0),   # Vertical
            (1, 1),   # Main diagonal
            (1, -1)   # Anti-diagonal
        ]

        for dr, dc in directions:
            count = 1  # Count the current position

            # Count in positive direction
            count += self._count_consecutive(row, col, dr, dc, player)

            # Count in negative direction
            count += self._count_consecutive(row, col, -dr, -dc, player)

            if count >= self.k:
                return True

        return False

    def _count_consecutive(self, row: int, col: int, dr: int, dc: int, player: int) -> int:
        """
        Count consecutive marks in one direction.
        """
        count = 0
        r, c = row + dr, col + dc

        while 0 <= r < self.rows and 0 <= c < self.cols and self.board[r][c] == player:
            count += 1
            r += dr
            c += dc

        return count

# Example usage
if __name__ == "__main__":
    print("=== Game with AI ===")
    game = TicTacToeWithAI(3, 3, 3, isAI=True)

    # Human player makes moves, AI responds automatically
    game.move(0, 0)  # Human (Player 1)
    # AI automatically responds

    game.move(0, 1)  # Human (Player 1)
    # AI automatically responds

    game.move(0, 2)  # Human (Player 1) tries to win
    # Game ends

=== Game with AI ===

Board:
| X |   |   |
|   |   |   |
|   |   |   |

AI (Player 2) is thinking...
AI plays at (2, 0)

Board:
| X |   |   |
|   |   |   |
| O |   |   |


Board:
| X | X |   |
|   |   |   |
| O |   |   |

AI (Player 2) is thinking...
AI blocks at (0, 2)

Board:
| X | X | O |
|   |   |   |
| O |   |   |


Board:
| X | X | X |
|   |   |   |
| O |   |   |

Player 1 won
